#### Question 1

In [12]:
import pandas as pd
import numpy as np


data = {
        'CustomerID': ['C001', 'C002', 'C003', 'C004', 'C005', 'C001', 'C006'],
        'Age': ['25', '30', 'not_available', '45', '25', '25', np.nan],
        'Gender': ['Male', 'Female', np.nan, 'Male', 'Female', 'Male', 'Female'],
        'Annual_Income': ['50,000', '60,000', 'not available', '75,000', '50,000', '50,000', '62,000'],
        'City': ['New York', 'Los Angeles', 'Chicago', np.nan, 'New York', 'New York', 'Houston']
}
df = pd.DataFrame(data)

print("\nOriginal missing values:\n", df.isnull().sum())

initial_rows = df.shape[0]

# Calculate the threshold: keep rows with at least (total_columns - 2) non-missing values
threshold = df.shape[1] - 2
df.dropna(thresh=threshold, inplace=True)
print(f"\nRemoved {initial_rows - df.shape[0]} rows with more than 2 missing values.")
print("Missing values after removing rows:\n", df.isnull().sum())

# Convert Age to numeric, coercing errors to NaN, before filling
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# 4. Fill missing Age using median
if not df['Age'].isnull().all(): # Check if there are any non-NaN values to calculate median
    age_median = df['Age'].median()
    df['Age'].fillna(age_median, inplace=True)
    print(f"\nFilled missing Age with median: {age_median}")
else:
    print("\nCannot calculate median for Age, as all values are NaN after conversion/cleaning.")

# 5. Fill missing City using mode
if not df['City'].isnull().all(): # Check if there are any non-NaN values to calculate mode
    city_mode = df['City'].mode()[0]
    df['City'].fillna(city_mode, inplace=True)
    print(f"Filled missing City with mode: {city_mode}")
else:
    print("\nCannot calculate mode for City, as all values are NaN after cleaning.")


# 6. Convert Annual_Income into numeric
# Remove commas and replace 'not available' with NaN
df['Annual_Income'] = df['Annual_Income'].astype(str).str.replace(',', '').replace('not available', np.nan)
# Convert to numeric, coercing errors to NaN
df['Annual_Income'] = pd.to_numeric(df['Annual_Income'], errors='coerce')
print("\nAnnual_Income converted to numeric. Missing values in Annual_Income:\n", df['Annual_Income'].isnull().sum())

# 7. Remove duplicate rows
initial_rows_after_na = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows_after_na - df.shape[0]} duplicate rows.")

# 8. Convert Gender into numerical form using appropriate encoding
df['Gender'] = df['Gender'].astype(str).replace('nan', np.nan) # Ensure np.nan for actual missing values

if df['Gender'].isnull().any():
    gender_mode = df['Gender'].mode()[0] if not df['Gender'].mode().empty else 'Unknown' # Fallback for empty mode
    df['Gender'].fillna(gender_mode, inplace=True)
    print(f"\nFilled missing Gender with mode: {gender_mode} before encoding.")

if 'Male' in df['Gender'].unique() or 'Female' in df['Gender'].unique():
    gender_mapping = {'Male': 0, 'Female': 1}
    df['Gender_encoded'] = df['Gender'].map(gender_mapping)
    print("\nGender column converted to numerical form (Male: 0, Female: 1).")
else:
    print("\nGender column does not contain 'Male' or 'Female' for encoding.")


print("\nCleaned DataFrame head:\n", df.head())
print("\nCleaned DataFrame info:\n")
df.info()
print("\nMissing values after all cleaning steps:\n", df.isnull().sum())


Original missing values:
 CustomerID       0
Age              1
Gender           1
Annual_Income    0
City             1
dtype: int64

Removed 0 rows with more than 2 missing values.
Missing values after removing rows:
 CustomerID       0
Age              1
Gender           1
Annual_Income    0
City             1
dtype: int64

Filled missing Age with median: 25.0
Filled missing City with mode: New York

Annual_Income converted to numeric. Missing values in Annual_Income:
 1

Removed 1 duplicate rows.

Filled missing Gender with mode: Female before encoding.

Gender column converted to numerical form (Male: 0, Female: 1).

Cleaned DataFrame head:
   CustomerID   Age  Gender  Annual_Income         City  Gender_encoded
0       C001  25.0    Male        50000.0     New York               0
1       C002  30.0  Female        60000.0  Los Angeles               1
2       C003  25.0  Female            NaN      Chicago               1
3       C004  45.0    Male        75000.0     New York      

/tmp/ipykernel_4915/3696363974.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(age_median, inplace=True)
/tmp/ipykernel_4915/3696363974.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

#### Question 2

In [13]:
import pandas as pd
import numpy as np

data = {
    'Patient_ID': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007'],
    'Admission_Date': ['01-02-2024', '2023/11/15', '10-01-2024', '2024-05-30', 'invalid-date', np.nan, '03-03-2024'],
    'Discharge_Date': ['05-02-2024', '2023/11/20', 'invalid-date', '2024-06-05', '12-12-2024', '2024/04/10', np.nan],
    'Severity': ['Low', 'Medium', 'High', 'Medium', 'Low', np.nan, 'High'],
    'Department': ['Cardiology', 'Neurology', 'Orthopedic', 'Cardiology', np.nan, 'Neurology', 'Orthopedic']
}

patients_df = pd.DataFrame(data)

print("\nInitial Patients DataFrame head:")
print(patients_df.head())
print("\nInitial DataFrame info:")
patients_df.info()

# 1. Convert Admission_Date and Discharge_Date to datetime format
# 2. Handle invalid dates using appropriate method (errors='coerce' will turn invalid into NaT)
print("\nConverting Date columns to datetime and coercing errors...")
patients_df['Admission_Date'] = pd.to_datetime(patients_df['Admission_Date'], errors='coerce', dayfirst=True)
patients_df['Discharge_Date'] = pd.to_datetime(patients_df['Discharge_Date'], errors='coerce', dayfirst=True)
print("Date columns converted. Invalid/Unparseable dates are now NaT (Not a Time).")
print("Missing/Invalid dates after conversion:")
print(patients_df[['Admission_Date', 'Discharge_Date']].isnull().sum())

print("\nDataFrame info after date conversion:")
patients_df.info()

# 3. Encode Severity using ordinal encoding
print("\nEncoding 'Severity' using ordinal encoding (Low: 0, Medium: 1, High: 2)...")
severity_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
patients_df['Severity_encoded'] = patients_df['Severity'].map(severity_mapping)
print("Severity encoded. Note: Missing Severity values will result in NaN in 'Severity_encoded'.")
print("NaNs in 'Severity_encoded':", patients_df['Severity_encoded'].isnull().sum())

# 4. Encode Department using one-hot encoding
print("\nEncoding 'Department' using one-hot encoding...")
patients_df = pd.get_dummies(patients_df, columns=['Department'], prefix='Department', dtype=int)
print("Department one-hot encoded.")

# 5. Check and fix any incorrect data types (already addressed throughout implicitly by conversions)
# Final check of data types after all transformations
print("\nFinal DataFrame head after all transformations:")
print(patients_df.head())
print("\nFinal DataFrame info after all transformations:")
patients_df.info()
print("\nMissing values after all transformations:")
print(patients_df.isnull().sum())


Initial Patients DataFrame head:
  Patient_ID Admission_Date Discharge_Date Severity  Department
0       P001     01-02-2024     05-02-2024      Low  Cardiology
1       P002     2023/11/15     2023/11/20   Medium   Neurology
2       P003     10-01-2024   invalid-date     High  Orthopedic
3       P004     2024-05-30     2024-06-05   Medium  Cardiology
4       P005   invalid-date     12-12-2024      Low         NaN

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Patient_ID      7 non-null      object
 1   Admission_Date  6 non-null      object
 2   Discharge_Date  6 non-null      object
 3   Severity        6 non-null      object
 4   Department      6 non-null      object
dtypes: object(5)
memory usage: 412.0+ bytes

Converting Date columns to datetime and coercing errors...
Date columns converted. Invalid/Unparseable 

#### Question 3

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

data = {
    'Area': np.random.randint(500, 5001, 100), # 500-5000 sq ft
    'Bedrooms': np.random.randint(1, 6, 100),   # 1-5 bedrooms
    'Price': np.random.randint(150000, 750001, 100), # Target variable
    'Distance_from_City': np.random.uniform(1, 50, 100).round(2), # 1-50 km
    'Age_of_House': np.random.randint(1, 70, 100) # 1-70 years
}
housing_df = pd.DataFrame(data)

print("\nInitial Housing DataFrame head:")
print(housing_df.head())
print("\nInitial DataFrame info:")
housing_df.info()

# 1. Identify which features need scaling
features_to_scale_normalize = ['Area', 'Distance_from_City']
features_to_scale_standardize = ['Age_of_House']

# Separate features (X) and target (y)
X = housing_df.drop('Price', axis=1)
y = housing_df['Price']

print(f"\nFeatures identified for Normalization: {features_to_scale_normalize}")
print(f"Features identified for Standardization: {features_to_scale_standardize}")

# 4. Train-test split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nData split into training (X_train shape: {X_train.shape}) and testing (X_test shape: {X_test.shape}) sets.")

# 2. Apply Normalization on Area and Distance_from_City
# 3. Apply Standardization on Age_of_House
# 5. Apply scaling correctly (fit on train, transform on test)

# Initialize scalers
min_max_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

print("\nApplying Normalization to 'Area' and 'Distance_from_City'...")
print("Applying Standardization to 'Age_of_House'...")

# Normalization: Fit on X_train, transform X_train and X_test
X_train[features_to_scale_normalize] = min_max_scaler.fit_transform(X_train[features_to_scale_normalize])
X_test[features_to_scale_normalize] = min_max_scaler.transform(X_test[features_to_scale_normalize])

# Standardization: Fit on X_train, transform X_train and X_test
X_train[features_to_scale_standardize] = standard_scaler.fit_transform(X_train[features_to_scale_standardize])
X_test[features_to_scale_standardize] = standard_scaler.transform(X_test[features_to_scale_standardize])

print("\nFeature scaling completed.")

print("\nScaled X_train head:")
print(X_train.head())

print("\nScaled X_test head:")
print(X_test.head())

print("\nDescriptive statistics for scaled 'Area' (train set):")
print(X_train['Area'].describe())

print("\nDescriptive statistics for scaled 'Age_of_House' (train set):")
print(X_train['Age_of_House'].describe())



Initial Housing DataFrame head:
   Area  Bedrooms   Price  Distance_from_City  Age_of_House
0  1371         3  643224               39.33            24
1  4364         1  161280                8.50            58
2  3399         4  265323               43.01             4
3  2203         1  739836               35.32            66
4  2626         5  435120                4.20            15

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Area                100 non-null    int64  
 1   Bedrooms            100 non-null    int64  
 2   Price               100 non-null    int64  
 3   Distance_from_City  100 non-null    float64
 4   Age_of_House        100 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 4.0 KB

Features identified for Normalization: ['Area', 'Distance_from_City']
Features identi

#### Question 4

In [15]:
import pandas as pd
import numpy as np

data = {
    'Order_ID': ['ORD001', 'ORD002', 'ORD003', 'ORD001', 'ORD004', 'ORD005', 'ORD006', 'ORD003'],
    'Restaurant_Name': ['Pizza Place', 'Burger Joint', 'Sushi Spot', 'Pizza Place', 'Taco Stand', 'Curry House', 'Sushi Spot', 'Sushi Spot'],
    'Cuisine_Type': ['Italian', 'American', 'Japanese', 'Italian', 'Mexican', 'Pakistani', 'Japanese', 'Japanese'],
    'Delivery_Time': [30, 25, np.nan, 30, 40, 35, 20, 28],
    'Customer_Rating': [4.5, 3.8, 5.0, 4.5, np.nan, 4.2, 4.9, 5.0]
}
orders_df = pd.DataFrame(data)

print("\nInitial Orders DataFrame head:")
print(orders_df.head())
print("\nInitial DataFrame info:")
orders_df.info()
print("\nInitial Missing values:\n", orders_df.isnull().sum())

# 1. Detect and count duplicate rows
duplicate_rows = orders_df.duplicated().sum()
print(f"\nNumber of duplicate rows detected: {duplicate_rows}")

# 2. Remove duplicate records
initial_rows = orders_df.shape[0]
orders_df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - orders_df.shape[0]} duplicate rows.")
print("DataFrame head after removing duplicates:\n", orders_df.head())

# 3. Fill missing Delivery_Time using mean
if not orders_df['Delivery_Time'].isnull().all():
    mean_delivery_time = orders_df['Delivery_Time'].mean()
    orders_df['Delivery_Time'].fillna(mean_delivery_time, inplace=True)
    print(f"\nFilled missing Delivery_Time with mean: {mean_delivery_time:.2f} minutes.")
else:
    print("\nDelivery_Time column is all NaN, cannot fill with mean.")

# 4. Fill missing Customer_Rating using median
if not orders_df['Customer_Rating'].isnull().all():
    median_customer_rating = orders_df['Customer_Rating'].median()
    orders_df['Customer_Rating'].fillna(median_customer_rating, inplace=True)
    print(f"Filled missing Customer_Rating with median: {median_customer_rating:.1f}.")
else:
    print("\nCustomer_Rating column is all NaN, cannot fill with median.")

# 5. Encode Cuisine_Type using one-hot encoding
print("\nEncoding 'Cuisine_Type' using one-hot encoding...")
orders_df = pd.get_dummies(orders_df, columns=['Cuisine_Type'], prefix='Cuisine', dtype=int)
print("Cuisine_Type one-hot encoded.")

# 6. Check final dataset consistency
print("\nFinal DataFrame head after all transformations:")
print(orders_df.head())
print("\nFinal DataFrame info after all transformations:")
orders_df.info()
print("\nMissing values after all transformations:\n", orders_df.isnull().sum())


Initial Orders DataFrame head:
  Order_ID Restaurant_Name Cuisine_Type  Delivery_Time  Customer_Rating
0   ORD001     Pizza Place      Italian           30.0              4.5
1   ORD002    Burger Joint     American           25.0              3.8
2   ORD003      Sushi Spot     Japanese            NaN              5.0
3   ORD001     Pizza Place      Italian           30.0              4.5
4   ORD004      Taco Stand      Mexican           40.0              NaN

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         8 non-null      object 
 1   Restaurant_Name  8 non-null      object 
 2   Cuisine_Type     8 non-null      object 
 3   Delivery_Time    7 non-null      float64
 4   Customer_Rating  7 non-null      float64
dtypes: float64(2), object(3)
memory usage: 452.0+ bytes

Initial Missing values:
 Order_

/tmp/ipykernel_4915/2260838680.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  orders_df['Delivery_Time'].fillna(mean_delivery_time, inplace=True)
/tmp/ipykernel_4915/2260838680.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value,

#### Question 5

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Eve', 'Frank', 'Grace', 'Charlie'],
    'Marks': ['85 marks', '72 marks', np.nan, '85 marks', '60', '90 marks', '75', 'not available', '72 marks'],
    'Grade': ['A', 'B', 'C', 'A', 'D', 'A', 'B', np.nan, 'B'],
    'Study_Hours': [5, 3, 4, 5, 2, 6, 3, 5, 4],
    'Passed': ['Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes']
}
students_df = pd.DataFrame(data)

print("\nInitial Students DataFrame head:")
print(students_df.head())
print("\nInitial DataFrame info:")
students_df.info()
print("\nInitial Missing values:\n", students_df.isnull().sum())

# 1. Clean Marks column and convert it to numeric
# Remove ' marks' and convert to numeric, coercing errors to NaN
students_df['Marks'] = students_df['Marks'].astype(str).str.replace(' marks', '', regex=False).replace('not available', np.nan)
students_df['Marks'] = pd.to_numeric(students_df['Marks'], errors='coerce')
print("\n'Marks' column cleaned and converted to numeric. Missing values now:")
print(students_df['Marks'].isnull().sum())

# 2. Handle missing values in Marks using mean
if not students_df['Marks'].isnull().all():
    mean_marks = students_df['Marks'].mean()
    students_df['Marks'].fillna(mean_marks, inplace=True)
    print(f"\nFilled missing 'Marks' with mean: {mean_marks:.2f}")
else:
    print("\n'Marks' column is all NaN, cannot fill with mean.")

# 3. Remove duplicate rows
initial_rows = students_df.shape[0]
students_df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - students_df.shape[0]} duplicate rows.")
print("DataFrame head after removing duplicates:\n", students_df.head())

# 4. Encode Grade using ordinal encoding
# Define the order for grades
grade_order = ['D', 'C', 'B', 'A']
ordinal_encoder = OrdinalEncoder(categories=[grade_order], dtype=int)
# Ensure 'Grade' column is treated as categorical with the defined order, fillna for consistent encoding
students_df['Grade_encoded'] = students_df['Grade'].astype('category').cat.set_categories(grade_order, ordered=True)
students_df['Grade_encoded'] = students_df['Grade_encoded'].cat.codes # .cat.codes converts categories to numerical codes (-1 for NaN)


# If you want to impute missing grades before encoding (e.g. with mode):
if students_df['Grade'].isnull().any():
    mode_grade = students_df['Grade'].mode()[0] if not students_df['Grade'].mode().empty else 'A'
    students_df['Grade'].fillna(mode_grade, inplace=True)
    print(f"\nFilled missing 'Grade' with mode: {mode_grade} before encoding.")

# Re-apply ordinal encoding after filling missing values
ordinal_encoder = OrdinalEncoder(categories=[grade_order], dtype=int, handle_unknown='use_encoded_value', unknown_value=-1)
students_df['Grade_encoded'] = ordinal_encoder.fit_transform(students_df[['Grade']])

print("\n'Grade' column encoded using ordinal encoding (D=0, C=1, B=2, A=3).")

# 5. Encode Passed using label encoding
label_encoder = LabelEncoder()
students_df['Passed_encoded'] = label_encoder.fit_transform(students_df['Passed'])
print("\n'Passed' column encoded using label encoding (e.g., No=0, Yes=1).")
print("Mapping for 'Passed':", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# 6. Apply standardization on Study_Hours
scaler = StandardScaler()
students_df['Study_Hours_scaled'] = scaler.fit_transform(students_df[['Study_Hours']])
print("\n'Study_Hours' column standardized.")

# Check final dataset consistency
print("\nFinal DataFrame head after all transformations:")
print(students_df.head())
print("\nFinal DataFrame info after all transformations:")
students_df.info()
print("\nMissing values after all transformations:\n", students_df.isnull().sum())


Initial Students DataFrame head:
      Name     Marks Grade  Study_Hours Passed
0    Alice  85 marks     A            5    Yes
1      Bob  72 marks     B            3    Yes
2  Charlie       NaN     C            4     No
3    Alice  85 marks     A            5    Yes
4    David        60     D            2     No

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         9 non-null      object
 1   Marks        8 non-null      object
 2   Grade        8 non-null      object
 3   Study_Hours  9 non-null      int64 
 4   Passed       9 non-null      object
dtypes: int64(1), object(4)
memory usage: 492.0+ bytes

Initial Missing values:
 Name           0
Marks          1
Grade          1
Study_Hours    0
Passed         0
dtype: int64

'Marks' column cleaned and converted to numeric. Missing values now:
2

Filled missing 'Mark

/tmp/ipykernel_4915/3597406968.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  students_df['Marks'].fillna(mean_marks, inplace=True)
/tmp/ipykernel_4915/3597406968.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

#### Question 6

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

data = {
    'Student_ID': [1, 2, 3, 4, 2],
    'Course': ['AI', 'Web Dev', 'AI', None, 'Web Dev'],
    'Completion_Percentage': ['80', '90%', None, '70', '90%'],
    'Enroll_Date': ['2024-01-10', '10/02/2024', 'invalid', '2024-03-01', '10/02/2024'],
    'Feedback': ['Good', 'Excellent', 'Average', 'Good', 'Excellent']
}
df = pd.DataFrame(data)

print("Initial DataFrame head:")
print(df.head())
print("\nInitial DataFrame info:")
df.info()

# 1. Detect missing values in the dataset
print("\nMissing values before cleaning:")
print(df.isnull().sum())

# 2. Remove duplicate rows
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - df.shape[0]} duplicate rows.")
print("DataFrame head after removing duplicates:")
print(df.head())

# 3. Clean Completion_Percentage (remove % and convert to numeric)
df['Completion_Percentage'] = df['Completion_Percentage'].astype(str).str.replace('%', '', regex=False)
df['Completion_Percentage'] = pd.to_numeric(df['Completion_Percentage'], errors='coerce')
print("\n'Completion_Percentage' cleaned and converted to numeric. Missing values now:")
print(df['Completion_Percentage'].isnull().sum())

# 4. Convert Enroll_Date into datetime and handle invalid values
# Use dayfirst=True to correctly parse '10/02/2024' as 10th February
df['Enroll_Date'] = pd.to_datetime(df['Enroll_Date'], errors='coerce', dayfirst=True)
print("\n'Enroll_Date' converted to datetime. Invalid dates are now NaT.")
print("Missing/Invalid 'Enroll_Date' after conversion:")
print(df['Enroll_Date'].isnull().sum())

# 5. Fill missing values:
# Completion_Percentage → mean
if not df['Completion_Percentage'].isnull().all():
    mean_completion = df['Completion_Percentage'].mean()
    df['Completion_Percentage'].fillna(mean_completion, inplace=True)
    print(f"\nFilled missing 'Completion_Percentage' with mean: {mean_completion:.2f}")
else:
    print("\n'Completion_Percentage' column is all NaN, cannot fill with mean.")

# Course → mode
if not df['Course'].isnull().all():
    mode_course = df['Course'].mode()[0]
    df['Course'].fillna(mode_course, inplace=True)
    print(f"Filled missing 'Course' with mode: {mode_course}")
else:
    print("\n'Course' column is all NaN, cannot fill with mode.")

# 6. Apply One-Hot Encoding on Course
df = pd.get_dummies(df, columns=['Course'], prefix='Course', dtype=int)
print("\n'Course' column one-hot encoded.")

# 7. Apply Label Encoding on Feedback
label_encoder = LabelEncoder()
df['Feedback_encoded'] = label_encoder.fit_transform(df['Feedback'])
print("\n'Feedback' column label encoded.")
print("Mapping for 'Feedback':", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# 8. Apply Normalization on Completion_Percentage
scaler = MinMaxScaler()
df['Completion_Percentage_normalized'] = scaler.fit_transform(df[['Completion_Percentage']])
print("\n'Completion_Percentage' column normalized.")

# Final consistency check
print("\nFinal DataFrame head after all transformations:")
print(df.head())
print("\nFinal DataFrame info after all transformations:")
df.info()
print("\nMissing values after all transformations:")
print(df.isnull().sum())

Initial DataFrame head:
   Student_ID   Course Completion_Percentage Enroll_Date   Feedback
0           1       AI                    80  2024-01-10       Good
1           2  Web Dev                   90%  10/02/2024  Excellent
2           3       AI                  None     invalid    Average
3           4     None                    70  2024-03-01       Good
4           2  Web Dev                   90%  10/02/2024  Excellent

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Student_ID             5 non-null      int64 
 1   Course                 4 non-null      object
 2   Completion_Percentage  4 non-null      object
 3   Enroll_Date            5 non-null      object
 4   Feedback               5 non-null      object
dtypes: int64(1), object(4)
memory usage: 332.0+ bytes

Missing values before cleaning

/tmp/ipykernel_4915/4073616495.py:47: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Completion_Percentage'].fillna(mean_completion, inplace=True)
/tmp/ipykernel_4915/4073616495.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, i

#### Question 7

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = {
    'Product_ID': [101, 102, 103, 104, 101],
    'Category': ['Electronics', 'Clothing', 'Electronics', 'Grocery', 'Electronics'],
    'Price': ['1000', '2000', 'invalid', '500', '1000'],
    'Quantity_Sold': [5, 10, None, 8, 5],
    'Discount': [10, 20, 15, None, 10],
    'Purchase_Date': ['2024/01/01', '01-02-2024', '2024-03-01', 'invalid', '2024/01/01']
}
df = pd.DataFrame(data)

print("Initial DataFrame head:")
print(df.head())
print("\nInitial DataFrame info:")
df.info()
print("\nInitial Missing values:\n", df.isnull().sum())

# 1. Identify and remove duplicate rows
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - df.shape[0]} duplicate rows.")
print("DataFrame head after removing duplicates:\n", df.head())

# 2. Convert Price into numeric and handle invalid values
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
print("\n'Price' column converted to numeric. Missing values now:")
print(df['Price'].isnull().sum())

# 3. Convert Purchase_Date into datetime and handle invalid values
# Use dayfirst=True to correctly parse '01-02-2024' as 1st February
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], errors='coerce', dayfirst=True)
print("\n'Purchase_Date' converted to datetime. Invalid dates are now NaT.")
print("Missing/Invalid 'Purchase_Date' after conversion:")
print(df['Purchase_Date'].isnull().sum())

# 4. Handle missing values:
if not df['Quantity_Sold'].isnull().all():
    median_quantity_sold = df['Quantity_Sold'].median()
    df['Quantity_Sold'].fillna(median_quantity_sold, inplace=True)
    print(f"\nFilled missing 'Quantity_Sold' with median: {median_quantity_sold:.2f}")
else:
    print("\n'Quantity_Sold' column is all NaN, cannot fill with median.")

if not df['Discount'].isnull().all():
    mean_discount = df['Discount'].mean()
    df['Discount'].fillna(mean_discount, inplace=True)
    print(f"Filled missing 'Discount' with mean: {mean_discount:.2f}")
else:
    print("\n'Discount' column is all NaN, cannot fill with mean.")

# 5. Apply One-Hot Encoding on Category
print("\nEncoding 'Category' using one-hot encoding...")
df = pd.get_dummies(df, columns=['Category'], prefix='Category', dtype=int)
print("'Category' one-hot encoded.")

# Prepare features (X) for train-test split and scaling
X = df.drop(columns=['Product_ID', 'Purchase_Date'])

# 6. Perform train-test split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
print(f"\nData split into training (X_train shape: {X_train.shape}) and testing (X_test shape: {X_test.shape}) sets.")

# 7. Apply Standardization on: Price, Quantity_Sold
features_to_standardize = ['Price', 'Quantity_Sold']
scaler = StandardScaler()

print("\nApplying Standardization to 'Price' and 'Quantity_Sold'...")

# Apply scaling correctly (fit on train, transform on test)
X_train[features_to_standardize] = scaler.fit_transform(X_train[features_to_standardize])
X_test[features_to_standardize] = scaler.transform(X_test[features_to_standardize])

print("Feature standardization completed.")

print("\nScaled X_train head:")
print(X_train.head())

print("\nScaled X_test head:")
print(X_test.head())

# Final consistency check on the original DataFrame (df) which now contains encoded categories
print("\nFinal DataFrame head (before split/scaling):")
print(df.head())
print("\nMissing values in df after imputation and encoding:\n", df.isnull().sum())

Initial DataFrame head:
   Product_ID     Category    Price  Quantity_Sold  Discount Purchase_Date
0         101  Electronics     1000            5.0      10.0    2024/01/01
1         102     Clothing     2000           10.0      20.0    01-02-2024
2         103  Electronics  invalid            NaN      15.0    2024-03-01
3         104      Grocery      500            8.0       NaN       invalid
4         101  Electronics     1000            5.0      10.0    2024/01/01

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Product_ID     5 non-null      int64  
 1   Category       5 non-null      object 
 2   Price          5 non-null      object 
 3   Quantity_Sold  4 non-null      float64
 4   Discount       4 non-null      float64
 5   Purchase_Date  5 non-null      object 
dtypes: float64(2), int64(1), object(3)
memory us

/tmp/ipykernel_4915/3362016839.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Quantity_Sold'].fillna(median_quantity_sold, inplace=True)
/tmp/ipykernel_4915/3362016839.py:50: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpl

#### Question 8

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

data = {
    'Customer_ID': ['C001', 'C002', 'C003', 'C004', 'C001', 'C005', 'C006', 'C007'],
    'Age': ['25', '30', 'unknown', '45', '25', np.nan, '50', '35'],
    'Income': ['50,000', '60,000', 'not applicable', '75,000', '50,000', '90,000', '62,000', '100,000'],
    'Loan_Status': ['Approved', 'Rejected', 'Approved', 'Approved', 'Approved', 'Rejected', 'Approved', 'Rejected'],
    'Credit_Score': [700, 650, np.nan, 720, 700, 680, np.nan, 750],
    'City': ['New York', 'Los Angeles', np.nan, 'Chicago', 'New York', 'Houston', 'Miami', 'Chicago']
}
df = pd.DataFrame(data)

print("Initial DataFrame head:")
print(df.head())
print("\nInitial DataFrame info:")
df.info()

# 1. Detect missing values in all columns
print("\nMissing values before cleaning:")
print(df.isnull().sum())

# 2. Remove duplicate records
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - df.shape[0]} duplicate rows.")
print("DataFrame head after removing duplicates:\n", df.head())

# 3. Convert Income into numeric format
df['Income'] = df['Income'].astype(str).str.replace(',', '', regex=False).replace('not applicable', np.nan)
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
print("\n'Income' column converted to numeric. Missing values now:")
print(df['Income'].isnull().sum())

# 4. Clean and convert Age into numeric (handle invalid values)
df['Age'] = df['Age'].astype(str).replace('unknown', np.nan)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
print("\n'Age' column cleaned and converted to numeric. Missing values now:")
print(df['Age'].isnull().sum())

# 5. Handle missing values:
if not df['Age'].isnull().all():
    median_age = df['Age'].median()
    df['Age'].fillna(median_age, inplace=True)
    print(f"\nFilled missing 'Age' with median: {median_age:.1f}")
else:
    print("\n'Age' column is all NaN, cannot fill with median.")

if not df['Credit_Score'].isnull().all():
    mean_credit_score = df['Credit_Score'].mean()
    df['Credit_Score'].fillna(mean_credit_score, inplace=True)
    print(f"Filled missing 'Credit_Score' with mean: {mean_credit_score:.2f}")
else:
    print("\n'Credit_Score' column is all NaN, cannot fill with mean.")

if not df['City'].isnull().all():
    mode_city = df['City'].mode()[0]
    df['City'].fillna(mode_city, inplace=True)
    print(f"Filled missing 'City' with mode: {mode_city}")
else:
    print("\n'City' column is all NaN, cannot fill with mode.")

# 6. Apply Label Encoding on Loan_Status
label_encoder = LabelEncoder()
df['Loan_Status_encoded'] = label_encoder.fit_transform(df['Loan_Status'])
print("\n'Loan_Status' column label encoded.")
print("Mapping for 'Loan_Status':", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# 7. Apply One-Hot Encoding on City
print("\nEncoding 'City' using one-hot encoding...")
df = pd.get_dummies(df, columns=['City'], prefix='City', dtype=int)
print("'City' one-hot encoded.")

# 8. Apply Standardization on Income and Credit_Score
features_to_standardize = ['Income', 'Credit_Score']
scaler = StandardScaler()
df[features_to_standardize] = scaler.fit_transform(df[features_to_standardize])
print("\n'Income' and 'Credit_Score' columns standardized.")

# Final consistency check
print("\nFinal DataFrame head after all transformations:")
print(df.head())
print("\nFinal DataFrame info after all transformations:")
df.info()
print("\nMissing values after all transformations:")
print(df.isnull().sum())

Initial DataFrame head:
  Customer_ID      Age          Income Loan_Status  Credit_Score         City
0        C001       25          50,000    Approved         700.0     New York
1        C002       30          60,000    Rejected         650.0  Los Angeles
2        C003  unknown  not applicable    Approved           NaN          NaN
3        C004       45          75,000    Approved         720.0      Chicago
4        C001       25          50,000    Approved         700.0     New York

Initial DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Customer_ID   8 non-null      object 
 1   Age           7 non-null      object 
 2   Income        8 non-null      object 
 3   Loan_Status   8 non-null      object 
 4   Credit_Score  6 non-null      float64
 5   City          7 non-null      object 
dtypes: float64(1), object(5)
memory us

/tmp/ipykernel_4915/762962793.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(median_age, inplace=True)
/tmp/ipykernel_4915/762962793.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 